
# Movimiento Circular


In [1]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación: Origen de la Aceleración Centrípeta</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">
        Utiliza los controles inferiores para mostrar u ocultar los vectores. Observa cómo, al hacer <strong>Δt → 0</strong>, la aceleración media encaja exactamente en la aceleración instantánea (hacia el centro).
    </p>
    
    <div style="display: flex; gap: 20px; margin-bottom: 20px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 15px; align-items: center;">
            <button id="play_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">▶ Reproducir Giro</button>
            <button id="limit_btn_UID" style="background-color: #6f42c1; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">🎯 Animar Límite (Δt → 0)</button>
        </div>
        
        <div style="display: flex; gap: 20px; align-items: center;">
            <div style="display: flex; flex-direction: column; align-items: center; background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc;">
                <label style="font-weight: 600; font-size: 0.85em; margin-bottom: 5px; color: #dc3545;">Intervalo de Tiempo (Δt)</label>
                <input type="range" id="dt_slider_UID" min="0.001" max="1.5" step="0.01" value="1.0" style="width: 150px;">
                <span id="dt_val_UID" style="font-weight: bold; font-family: monospace; font-size: 0.9em;">1.000 s</span>
            </div>
        </div>
    </div>
    
    <div style="display: flex; justify-content: center; margin-bottom: 15px;">
        <canvas id="canvas_sim_UID" width="550" height="450" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
    
    <div style="font-size: 0.95em; padding: 15px; background: #e2e3e5; border-radius: 5px; color: #383d41; display: flex; flex-wrap: wrap; gap: 15px; justify-content: center; user-select: none;">
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_v1_UID" checked style="cursor: pointer;">
            <span><b style="color:#0d6efd;">▬ v₁</b> (Vel. actual)</span>
        </label>
        
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_v2_UID" checked style="cursor: pointer;">
            <span><b style="color:#0dcaf0;">▬ v₂</b> (Vel. futura)</span>
        </label>
        
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_dv_UID" style="cursor: pointer;">
            <span><b style="color:#fd7e14;">▬ Δv</b> (Resta de vectores)</span>
        </label>
        
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_am_UID" style="cursor: pointer;">
            <span><b style="color:#6f42c1;">▬ a_media</b> (Δv/Δt)</span>
        </label>
        
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_ac_UID" style="cursor: pointer;">
            <span><b style="color:#dc3545;">▬ a_instantánea</b> (Real)</span>
        </label>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_sim_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_play = document.getElementById('play_UID');
            const btn_limit = document.getElementById('limit_btn_UID');
            const dt_slider = document.getElementById('dt_slider_UID');
            const dt_val = document.getElementById('dt_val_UID');
            
            // Elementos de visibilidad (Checkboxes)
            const chk_v1 = document.getElementById('chk_v1_UID');
            const chk_v2 = document.getElementById('chk_v2_UID');
            const chk_dv = document.getElementById('chk_dv_UID');
            const chk_am = document.getElementById('chk_am_UID');
            const chk_ac = document.getElementById('chk_ac_UID');
            
            let isPlaying = false;
            let limitAnimationActive = false;
            let animationId;
            let lastTime = 0; 
            
            // Físicas Base Fijas 
            const r = 2.0;       // Radio
            const omega = 1.0;   // Velocidad angular (1 rad/s)
            const v = omega * r;
            const ac = omega * omega * r;
            
            let t = 0;           // Tiempo actual
            let dt = parseFloat(dt_slider.value); // Intervalo delta t
            
            // Función para dibujar flechas
            function drawArrow(context, fromx, fromy, tox, toy, color, lineWidth=3, dashed=false) {
                const headlen = 12; // Aumentado ligeramente para balancear los vectores más grandes
                const angle = Math.atan2(toy - fromy, tox - fromx);
                
                context.beginPath();
                context.strokeStyle = color;
                context.lineWidth = lineWidth;
                if(dashed) context.setLineDash([5, 5]);
                else context.setLineDash([]);
                
                context.moveTo(fromx, fromy);
                context.lineTo(tox, toy);
                context.stroke();
                context.setLineDash([]); 
                
                context.beginPath();
                context.fillStyle = color;
                context.moveTo(tox, toy);
                context.lineTo(tox - headlen * Math.cos(angle - Math.PI / 6), toy - headlen * Math.sin(angle - Math.PI / 6));
                context.lineTo(tox - headlen * Math.cos(angle + Math.PI / 6), toy - headlen * Math.sin(angle + Math.PI / 6));
                context.fill();
            }

            function drawSim() {
                // Cálculos Físicos
                const t2 = t + dt;
                const theta1 = omega * t;
                const theta2 = omega * t2;
                
                // Posiciones reales
                const x1 = r * Math.cos(theta1),  y1 = r * Math.sin(theta1);
                const x2 = r * Math.cos(theta2),  y2 = r * Math.sin(theta2);
                
                // Velocidades 
                const v1x = -v * Math.sin(theta1), v1y = v * Math.cos(theta1);
                const v2x = -v * Math.sin(theta2), v2y = v * Math.cos(theta2);
                
                // Variación de velocidad y aceleración media
                const dvx = v2x - v1x, dvy = v2y - v1y;
                const amx = dvx / dt,  amy = dvy / dt;
                
                // Aceleración instantánea exacta
                const acx = -ac * Math.cos(theta1), acy = -ac * Math.sin(theta1);
                
                // Renderizado Canvas
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                const cx = canvas.width / 2;
                const cy = canvas.height / 2;
                
                // Factores de escala visuales (¡Aumentados para vectores mucho más grandes!)
                const Sp = 140 / r;           // Escala posición de la órbita
                const Sv = 110 / v;           // Escala velocidad (antes 50)
                const Sa = 120 / ac;          // Escala aceleración (antes 60)

                // Dibujar Ejes de Referencia
                ctx.strokeStyle = "#eee"; ctx.lineWidth = 1;
                ctx.beginPath(); ctx.moveTo(0, cy); ctx.lineTo(canvas.width, cy);
                ctx.moveTo(cx, 0); ctx.lineTo(cx, canvas.height); ctx.stroke();

                // Trayectoria Circular
                ctx.beginPath(); ctx.strokeStyle = "#aaa"; ctx.setLineDash([4, 4]);
                ctx.arc(cx, cy, r * Sp, 0, 2 * Math.PI); ctx.stroke(); ctx.setLineDash([]);

                // Coordenadas transformadas al canvas (Eje Y invertido)
                const c_x1 = cx + x1 * Sp, c_y1 = cy - y1 * Sp;
                const c_x2 = cx + x2 * Sp, c_y2 = cy - y2 * Sp;

                // Coordenadas de las puntas de los vectores
                const p_v1_x = c_x1 + v1x * Sv, p_v1_y = c_y1 - v1y * Sv;
                const p_v2_x = c_x2 + v2x * Sv, p_v2_y = c_y2 - v2y * Sv;
                const p_v2_trans_x = c_x1 + v2x * Sv, p_v2_trans_y = c_y1 - v2y * Sv; // v2 trasladado a pos 1

                // 1. Dibujar Aceleración Instantánea
                if (chk_ac.checked) {
                    drawArrow(ctx, c_x1, c_y1, c_x1 + acx * Sa, c_y1 - acy * Sa, "#dc3545", 3);
                }

                // 2. Dibujar Aceleración Media
                if (chk_am.checked) {
                    drawArrow(ctx, c_x1, c_y1, c_x1 + amx * Sa, c_y1 - amy * Sa, "#6f42c1", 3);
                }

                if (dt > 0.01) {
                    // Dibujar v2 en la posición 2
                    if (chk_v2.checked) {
                        drawArrow(ctx, c_x2, c_y2, p_v2_x, p_v2_y, "#0dcaf0", 3);
                        // Bolita 2 (Sombra)
                        ctx.beginPath(); ctx.fillStyle = "rgba(0,0,0,0.2)";
                        ctx.arc(c_x2, c_y2, 7, 0, Math.PI * 2); ctx.fill();
                    }
                    
                    // Dibujar Traslación de v2 y Resta Vectorial (Δv)
                    if (chk_dv.checked) {
                        drawArrow(ctx, c_x1, c_y1, p_v2_trans_x, p_v2_trans_y, "#0dcaf0", 3, true);
                        drawArrow(ctx, p_v1_x, p_v1_y, p_v2_trans_x, p_v2_trans_y, "#fd7e14", 3);
                    }
                }

                // 3. Dibujar v1 en la posición 1
                if (chk_v1.checked) {
                    drawArrow(ctx, c_x1, c_y1, p_v1_x, p_v1_y, "#0d6efd", 3);
                }

                // Bolita 1 (Posición actual, siempre visible)
                ctx.beginPath(); ctx.fillStyle = "#333";
                ctx.arc(c_x1, c_y1, 10, 0, Math.PI * 2); ctx.fill();
            }

            // Actualizar canvas cuando cambia un checkbox (si no está animándose)
            [chk_v1, chk_v2, chk_dv, chk_am, chk_ac].forEach(chk => {
                chk.addEventListener('change', () => {
                    if (!isPlaying && !limitAnimationActive) drawSim();
                });
            });

            function animate(timestamp) {
                if (!lastTime) lastTime = timestamp;
                let deltaTime = (timestamp - lastTime) / 1000;
                lastTime = timestamp;
                
                if (isPlaying) {
                    t += deltaTime * 0.5; // Velocidad de rotación
                }
                
                if (limitAnimationActive) {
                    dt = dt * 0.99; // Límite suave
                    if (dt <= 0.005) {
                        dt = 0.001; 
                        limitAnimationActive = false;
                        btn_limit.innerText = "🎯 Animar Límite (Δt → 0)";
                        btn_limit.style.backgroundColor = "#6f42c1";
                    }
                    dt_slider.value = dt;
                    dt_val.innerText = dt.toFixed(3) + " s";
                }
                
                drawSim();
                
                if(isPlaying || limitAnimationActive) {
                    animationId = requestAnimationFrame(animate);
                }
            }

            dt_slider.addEventListener('input', () => {
                dt = parseFloat(dt_slider.value);
                dt_val.innerText = dt.toFixed(3) + " s";
                limitAnimationActive = false;
                btn_limit.innerText = "🎯 Animar Límite (Δt → 0)";
                btn_limit.style.backgroundColor = "#6f42c1";
                if(!isPlaying && !limitAnimationActive) drawSim();
            });

            btn_play.addEventListener('click', () => {
                isPlaying = !isPlaying;
                if (isPlaying) {
                    btn_play.innerText = "⏸ Pausar Giro";
                    btn_play.style.backgroundColor = "#ffc107";
                    btn_play.style.color = "#000";
                    lastTime = 0; 
                    animationId = requestAnimationFrame(animate);
                } else {
                    btn_play.innerText = "▶ Reproducir Giro";
                    btn_play.style.backgroundColor = "#198754";
                    btn_play.style.color = "#fff";
                }
            });

            btn_limit.addEventListener('click', () => {
                if(dt <= 0.01) {
                    dt = 1.0;
                    dt_slider.value = dt;
                }
                limitAnimationActive = true;
                btn_limit.innerText = "⏳ Acercando...";
                btn_limit.style.backgroundColor = "#6c757d";
                lastTime = 0;
                
                if (!isPlaying) animationId = requestAnimationFrame(animate);
            });
            
            drawSim();
        }
        
        initSimulation();
    })();
</script>
"""

# Reemplazamos el identificador para evitar conflictos y mostramos
html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))

In [2]:
import random
from IPython.display import display, HTML

uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50; text-align: center;">Simulación: Movimiento Circular Acelerado</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px; text-align: center;">
        Observa cómo las líneas segmentadas grises prolongan los vectores. Nota que <strong>solo la aceleración centrípeta apunta exactamente al centro</strong> de la trayectoria, mientras que la aceleración total se desvía debido a la aceleración tangencial.
    </p>
    
    <div style="display: flex; justify-content: center; flex-wrap: wrap; gap: 15px; margin-bottom: 20px;">
        <button id="btn_reset_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; width: 180px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); font-size: 1.05em;">▶ Iniciar / Reiniciar</button>
        <button id="btn_pause_UID" style="background-color: #ffc107; color: black; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; width: 180px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); font-size: 1.05em;">⏸ Pausar</button>
        <button id="btn_mute_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; width: 150px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); font-size: 1.05em;">🔊 Silenciar</button>
    </div>
    
    <div style="display: flex; justify-content: center; margin-bottom: 15px; position: relative;">
        <canvas id="canvas_mcu_UID" width="550" height="450" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
    
    <div style="font-size: 0.95em; padding: 15px; background: #e2e3e5; border-radius: 5px; color: #383d41; display: flex; flex-wrap: wrap; gap: 15px; justify-content: center; user-select: none;">
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_v_UID" checked>
            <span><b style="color:#20c997;">▬ v</b> (Vel. Lineal)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_ac_UID">
            <span><b style="color:#dc3545;">▬ a_c</b> (Centrípeta)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_at_UID">
            <span><b style="color:#fd7e14;">▬ a_t</b> (Tangencial)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_atot_UID">
            <span><b style="color:#6f42c1;">▬ a_tot</b> (Total)</span>
        </label>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_mcu_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_reset = document.getElementById('btn_reset_UID');
            const btn_pause = document.getElementById('btn_pause_UID');
            const btn_mute = document.getElementById('btn_mute_UID');
            
            const chk_v = document.getElementById('chk_v_UID');
            const chk_ac = document.getElementById('chk_ac_UID');
            const chk_at = document.getElementById('chk_at_UID');
            const chk_atot = document.getElementById('chk_atot_UID');
            
            let lastTime = 0;
            let isRunning = false;
            let isPaused = false;
            let wasAccelerating = true;
            let isMuted = false;
            
            const AudioContext = window.AudioContext || window.webkitAudioContext;
            let audioCtx = null;
            let oscillator = null;
            let filterNode = null;
            let gainNode = null;
            
            const r_fisico = 2.0; 
            let theta = 0;
            let omega = 0.1;  
            let alpha = 0.15; 
            const MAX_OMEGA = 1.7; 
            const TIME_SCALE = 0.8; 
            
            function initAudio() {
                if (!audioCtx) audioCtx = new AudioContext();
                if (audioCtx.state === 'suspended') audioCtx.resume();
                
                if (oscillator) {
                    oscillator.stop();
                    oscillator.disconnect();
                }
                
                oscillator = audioCtx.createOscillator();
                filterNode = audioCtx.createBiquadFilter();
                gainNode = audioCtx.createGain();
                
                oscillator.type = 'sawtooth'; 
                oscillator.frequency.value = 30; 
                
                filterNode.type = 'lowpass';
                filterNode.frequency.value = 100; 
                
                gainNode.gain.value = isMuted ? 0 : 0.25; 
                
                oscillator.connect(filterNode);
                filterNode.connect(gainNode);
                gainNode.connect(audioCtx.destination);
                oscillator.start();
            }

            function updateAudio(currentOmega, forceExact = false) {
                if (oscillator && audioCtx && filterNode) {
                    const targetFreq = 30 + (currentOmega * 30);
                    const filterTarget = 100 + (currentOmega * 50);
                    
                    if (forceExact) {
                        oscillator.frequency.setValueAtTime(targetFreq, audioCtx.currentTime);
                        filterNode.frequency.setValueAtTime(filterTarget, audioCtx.currentTime);
                    } else {
                        oscillator.frequency.setTargetAtTime(targetFreq, audioCtx.currentTime, 0.1);
                        filterNode.frequency.setTargetAtTime(filterTarget, audioCtx.currentTime, 0.1);
                    }
                }
            }

            function drawArrow(context, fromx, fromy, tox, toy, color, lineWidth=3, dashed=false) {
                const headlen = 12;
                const dx = tox - fromx;
                const dy = toy - fromy;
                const length = Math.sqrt(dx*dx + dy*dy);
                
                if(length < 0.1) return; 
                
                const angle = Math.atan2(dy, dx);
                
                context.beginPath();
                context.strokeStyle = color;
                context.lineWidth = lineWidth;
                if(dashed) context.setLineDash([5, 5]);
                else context.setLineDash([]);
                
                context.moveTo(fromx, fromy);
                context.lineTo(tox, toy);
                context.stroke();
                context.setLineDash([]); 
                
                context.beginPath();
                context.fillStyle = color;
                context.moveTo(tox, toy);
                context.lineTo(tox - headlen * Math.cos(angle - Math.PI / 6), toy - headlen * Math.sin(angle - Math.PI / 6));
                context.lineTo(tox - headlen * Math.cos(angle + Math.PI / 6), toy - headlen * Math.sin(angle + Math.PI / 6));
                context.fill();
            }

            function drawProjection(fromX, fromY, dirX, dirY, color) {
                const extLen = 350; 
                ctx.beginPath(); 
                ctx.strokeStyle = color; 
                ctx.lineWidth = 1.5; 
                ctx.setLineDash([6, 6]);
                ctx.moveTo(fromX, fromY);
                ctx.lineTo(fromX + dirX * extLen, fromY + dirY * extLen);
                ctx.stroke(); 
                ctx.setLineDash([]);
            }

            function animate(timestamp) {
                if (!isRunning) {
                    lastTime = timestamp;
                    requestAnimationFrame(animate);
                    return;
                }
                
                if (isPaused) {
                    // Si está en pausa, solo actualizamos el tiempo para que no salte al reanudar
                    lastTime = timestamp;
                    requestAnimationFrame(animate);
                    return;
                }
                
                if (!lastTime) lastTime = timestamp;
                let dt = ((timestamp - lastTime) / 1000) * TIME_SCALE;
                lastTime = timestamp;
                
                if (omega >= MAX_OMEGA && wasAccelerating) {
                    alpha = 0; 
                    omega = MAX_OMEGA; 
                    wasAccelerating = false;
                    updateAudio(omega, true); 
                } else if (wasAccelerating) {
                    updateAudio(omega, false); 
                }
                
                omega += alpha * dt;
                theta += omega * dt;
                
                const v = omega * r_fisico;
                const ac = omega * omega * r_fisico;
                const at = alpha * r_fisico;
                
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                const cx = canvas.width / 2;
                const cy = canvas.height / 2;
                const R_visual = 130; 
                
                ctx.font = "bold 22px sans-serif";
                ctx.textAlign = "center";
                if (wasAccelerating) {
                    ctx.fillStyle = "#fd7e14";
                    ctx.fillText("ESTADO: ¡ACELERANDO!", cx, 40);
                } else {
                    ctx.fillStyle = "#198754";
                    ctx.fillText("ESTADO: RAPIDEZ CONSTANTE", cx, 40);
                }
                
                ctx.strokeStyle = "#eee"; ctx.lineWidth = 1;
                ctx.beginPath(); ctx.moveTo(0, cy); ctx.lineTo(canvas.width, cy);
                ctx.moveTo(cx, 0); ctx.lineTo(cx, canvas.height); ctx.stroke();
                
                ctx.beginPath(); ctx.strokeStyle = "#aaa"; ctx.setLineDash([4, 4]);
                ctx.arc(cx, cy, R_visual, 0, 2 * Math.PI); ctx.stroke(); ctx.setLineDash([]);

                const ur_x = Math.cos(theta);
                const ur_y = -Math.sin(theta);
                const ut_x = -Math.sin(theta);
                const ut_y = -Math.cos(theta);

                const px = cx + R_visual * ur_x;
                const py = cy + R_visual * ur_y;

                const scale_v = 30; 
                const scale_ac = 35; 
                const scale_at = 150; 

                const vx = ut_x * v * scale_v;
                const vy = ut_y * v * scale_v;
                
                const acx = -ur_x * ac * scale_ac; 
                const acy = -ur_y * ac * scale_ac;
                
                const atx = ut_x * at * scale_at;
                const aty = ut_y * at * scale_at;
                
                const atotx = acx + atx;
                const atoty = acy + aty;
                
                if (chk_ac.checked) {
                    drawProjection(px + acx, py + acy, -ur_x, -ur_y, "#aaa");
                }
                if (chk_at.checked && wasAccelerating) {
                    drawProjection(px + atx, py + aty, ut_x, ut_y, "#aaa");
                }
                if (chk_atot.checked) {
                    const normAtot = Math.sqrt(atotx*atotx + atoty*atoty);
                    if (normAtot > 0.1) {
                        drawProjection(px + atotx, py + atoty, atotx/normAtot, atoty/normAtot, "#aaa");
                    }
                }

                if (chk_v.checked) drawArrow(ctx, px, py, px + vx, py + vy, "#20c997", 4);
                if (chk_ac.checked) drawArrow(ctx, px, py, px + acx, py + acy, "#dc3545", 4);
                
                if (chk_at.checked && wasAccelerating) {
                    drawArrow(ctx, px, py, px + atx, py + aty, "#fd7e14", 4);
                }
                
                if (chk_atot.checked) {
                    drawArrow(ctx, px, py, px + atotx, py + atoty, "#6f42c1", 4);
                    
                    if (chk_ac.checked && chk_at.checked && wasAccelerating) {
                        drawArrow(ctx, px + acx, py + acy, px + atotx, py + atoty, "#ddd", 1, true);
                        drawArrow(ctx, px + atx, py + aty, px + atotx, py + atoty, "#ddd", 1, true);
                    }
                }

                ctx.beginPath(); ctx.fillStyle = "#333";
                ctx.arc(px, py, 12, 0, Math.PI * 2); ctx.fill();

                requestAnimationFrame(animate);
            }
            
            btn_reset.addEventListener('click', () => {
                initAudio();
                isRunning = true;
                isPaused = false;
                wasAccelerating = true;
                theta = 0;
                omega = 0.1;
                alpha = 0.15;
                
                btn_reset.innerText = "🔄 Reiniciar Animación";
                btn_pause.innerText = "⏸ Pausar";
                btn_pause.style.backgroundColor = "#ffc107";
                btn_pause.style.color = "black";
            });
            
            btn_pause.addEventListener('click', () => {
                if (!isRunning) return; // No hace nada si la simulación no ha iniciado
                
                isPaused = !isPaused;
                if (isPaused) {
                    btn_pause.innerText = "▶ Continuar";
                    btn_pause.style.backgroundColor = "#0d6efd";
                    btn_pause.style.color = "white";
                    if (audioCtx && audioCtx.state === 'running') {
                        audioCtx.suspend();
                    }
                } else {
                    btn_pause.innerText = "⏸ Pausar";
                    btn_pause.style.backgroundColor = "#ffc107";
                    btn_pause.style.color = "black";
                    if (audioCtx && audioCtx.state === 'suspended' && !isMuted) {
                        audioCtx.resume();
                    }
                }
            });
            
            btn_mute.addEventListener('click', () => {
                isMuted = !isMuted;
                if (gainNode) {
                    gainNode.gain.value = isMuted ? 0 : 0.25;
                }
                btn_mute.innerText = isMuted ? "🔇 Activar Sonido" : "🔊 Silenciar";
                btn_mute.style.backgroundColor = isMuted ? "#dc3545" : "#6c757d";
            });
            
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.font = "18px sans-serif";
            ctx.fillStyle = "#666";
            ctx.textAlign = "center";
            ctx.fillText("Presiona Iniciar para comenzar", canvas.width/2, canvas.height/2);

            requestAnimationFrame(animate);
        }
        
        initSimulation();
    })();
</script>
"""

html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))

In [3]:
import random
from IPython.display import display, HTML

uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50; text-align: center;">Simulación: Movimiento Circular Retardado (Frenado)</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px; text-align: center;">
        Observa cómo, al momento de frenar, la <strong>aceleración tangencial</strong> se opone a la velocidad (apunta hacia atrás), haciendo que la rapidez y la aceleración centrípeta disminuyan hasta detener el cuerpo.
    </p>
    
    <div style="display: flex; justify-content: center; flex-wrap: wrap; gap: 15px; margin-bottom: 20px;">
        <button id="btn_reset_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; width: 180px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); font-size: 1.05em;">▶ Iniciar / Reiniciar</button>
        <button id="btn_pause_UID" style="background-color: #ffc107; color: black; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; width: 180px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); font-size: 1.05em;">⏸ Pausar</button>
        <button id="btn_mute_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; width: 150px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); font-size: 1.05em;">🔊 Silenciar</button>
    </div>
    
    <div style="display: flex; justify-content: center; margin-bottom: 15px; position: relative;">
        <canvas id="canvas_mcu_UID" width="550" height="450" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
    
    <div style="font-size: 0.95em; padding: 15px; background: #e2e3e5; border-radius: 5px; color: #383d41; display: flex; flex-wrap: wrap; gap: 15px; justify-content: center; user-select: none;">
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_v_UID" checked>
            <span><b style="color:#20c997;">▬ v</b> (Vel. Lineal)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_ac_UID">
            <span><b style="color:#dc3545;">▬ a_c</b> (Centrípeta)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_at_UID">
            <span><b style="color:#fd7e14;">▬ a_t</b> (Tangencial)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_atot_UID">
            <span><b style="color:#6f42c1;">▬ a_tot</b> (Total)</span>
        </label>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_mcu_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_reset = document.getElementById('btn_reset_UID');
            const btn_pause = document.getElementById('btn_pause_UID');
            const btn_mute = document.getElementById('btn_mute_UID');
            
            const chk_v = document.getElementById('chk_v_UID');
            const chk_ac = document.getElementById('chk_ac_UID');
            const chk_at = document.getElementById('chk_at_UID');
            const chk_atot = document.getElementById('chk_atot_UID');
            
            let lastTime = 0;
            let isRunning = false;
            let isPaused = false;
            let isMuted = false;
            
            // Fases de simulación: 1 (Constante), 2 (Frenando), 3 (Detenido)
            let phase = 1; 
            
            const AudioContext = window.AudioContext || window.webkitAudioContext;
            let audioCtx = null;
            let oscillator = null;
            let filterNode = null;
            let gainNode = null;
            
            const r_fisico = 2.0; 
            let theta = 0;
            const INITIAL_OMEGA = 1.5;
            let omega = INITIAL_OMEGA;  
            let alpha = 0; 
            const TIME_SCALE = 0.8; 
            
            function initAudio() {
                if (!audioCtx) audioCtx = new AudioContext();
                if (audioCtx.state === 'suspended') audioCtx.resume();
                
                if (oscillator) {
                    oscillator.stop();
                    oscillator.disconnect();
                }
                
                oscillator = audioCtx.createOscillator();
                filterNode = audioCtx.createBiquadFilter();
                gainNode = audioCtx.createGain();
                
                oscillator.type = 'sawtooth'; 
                oscillator.frequency.value = 30 + (INITIAL_OMEGA * 30); 
                
                filterNode.type = 'lowpass';
                filterNode.frequency.value = 100 + (INITIAL_OMEGA * 50); 
                
                gainNode.gain.value = isMuted ? 0 : 0.25; 
                
                oscillator.connect(filterNode);
                filterNode.connect(gainNode);
                gainNode.connect(audioCtx.destination);
                oscillator.start();
            }

            function updateAudio(currentOmega) {
                if (oscillator && audioCtx && filterNode) {
                    const targetFreq = 30 + (currentOmega * 30);
                    const filterTarget = 100 + (currentOmega * 50);
                    
                    oscillator.frequency.setTargetAtTime(targetFreq, audioCtx.currentTime, 0.1);
                    filterNode.frequency.setTargetAtTime(filterTarget, audioCtx.currentTime, 0.1);
                }
            }

            function drawArrow(context, fromx, fromy, tox, toy, color, lineWidth=3, dashed=false) {
                const headlen = 12;
                const dx = tox - fromx;
                const dy = toy - fromy;
                const length = Math.sqrt(dx*dx + dy*dy);
                
                if(length < 0.1) return; 
                
                const angle = Math.atan2(dy, dx);
                
                context.beginPath();
                context.strokeStyle = color;
                context.lineWidth = lineWidth;
                if(dashed) context.setLineDash([5, 5]);
                else context.setLineDash([]);
                
                context.moveTo(fromx, fromy);
                context.lineTo(tox, toy);
                context.stroke();
                context.setLineDash([]); 
                
                context.beginPath();
                context.fillStyle = color;
                context.moveTo(tox, toy);
                context.lineTo(tox - headlen * Math.cos(angle - Math.PI / 6), toy - headlen * Math.sin(angle - Math.PI / 6));
                context.lineTo(tox - headlen * Math.cos(angle + Math.PI / 6), toy - headlen * Math.sin(angle + Math.PI / 6));
                context.fill();
            }

            function drawProjection(fromX, fromY, dirX, dirY, color) {
                const extLen = 350; 
                ctx.beginPath(); 
                ctx.strokeStyle = color; 
                ctx.lineWidth = 1.5; 
                ctx.setLineDash([6, 6]);
                ctx.moveTo(fromX, fromY);
                ctx.lineTo(fromX + dirX * extLen, fromY + dirY * extLen);
                ctx.stroke(); 
                ctx.setLineDash([]);
            }

            function animate(timestamp) {
                if (!isRunning) {
                    lastTime = timestamp;
                    requestAnimationFrame(animate);
                    return;
                }
                
                if (isPaused) {
                    lastTime = timestamp;
                    requestAnimationFrame(animate);
                    return;
                }
                
                if (!lastTime) lastTime = timestamp;
                let dt = ((timestamp - lastTime) / 1000) * TIME_SCALE;
                lastTime = timestamp;
                
                // Lógica de fases para frenado
                if (phase === 1) {
                    // Después de 2 vueltas completas, comienza a frenar
                    if (theta >= 4 * Math.PI) {
                        phase = 2;
                        alpha = -0.08; // Frenado constante negativo
                    }
                } else if (phase === 2) {
                    // Si se detiene por completo
                    if (omega <= 0) {
                        phase = 3;
                        omega = 0;
                        alpha = 0;
                        if (gainNode && !isMuted) {
                            // Fade out suave al detenerse
                            gainNode.gain.setTargetAtTime(0, audioCtx.currentTime, 0.3);
                        }
                    }
                }
                
                omega += alpha * dt;
                if (omega < 0) omega = 0; // Evitar que empiece a girar hacia atrás
                theta += omega * dt;
                
                if (phase !== 3) updateAudio(omega); 
                
                const v = omega * r_fisico;
                const ac = omega * omega * r_fisico;
                const at = alpha * r_fisico;
                
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                const cx = canvas.width / 2;
                const cy = canvas.height / 2;
                const R_visual = 130; 
                
                // Texto superior adaptativo
                ctx.font = "bold 22px sans-serif";
                ctx.textAlign = "center";
                if (phase === 1) {
                    ctx.fillStyle = "#198754"; // Verde
                    ctx.fillText("ESTADO: RAPIDEZ CONSTANTE", cx, 40);
                } else if (phase === 2) {
                    ctx.fillStyle = "#dc3545"; // Rojo
                    ctx.fillText("ESTADO: ¡FRENANDO!", cx, 40);
                } else {
                    ctx.fillStyle = "#6c757d"; // Gris
                    ctx.fillText("ESTADO: DETENIDO", cx, 40);
                }
                
                ctx.strokeStyle = "#eee"; ctx.lineWidth = 1;
                ctx.beginPath(); ctx.moveTo(0, cy); ctx.lineTo(canvas.width, cy);
                ctx.moveTo(cx, 0); ctx.lineTo(cx, canvas.height); ctx.stroke();
                
                ctx.beginPath(); ctx.strokeStyle = "#aaa"; ctx.setLineDash([4, 4]);
                ctx.arc(cx, cy, R_visual, 0, 2 * Math.PI); ctx.stroke(); ctx.setLineDash([]);

                const ur_x = Math.cos(theta);
                const ur_y = -Math.sin(theta);
                const ut_x = -Math.sin(theta);
                const ut_y = -Math.cos(theta);

                const px = cx + R_visual * ur_x;
                const py = cy + R_visual * ur_y;

                // Escalas de dibujo (Tangencial exagerada para que destaque)
                const scale_v = 30; 
                const scale_ac = 35; 
                const scale_at = 200; 

                const vx = ut_x * v * scale_v;
                const vy = ut_y * v * scale_v;
                
                const acx = -ur_x * ac * scale_ac; 
                const acy = -ur_y * ac * scale_ac;
                
                // La matemática invierte automáticamente atx/aty porque 'alpha' es negativo
                const atx = ut_x * at * scale_at;
                const aty = ut_y * at * scale_at;
                
                const atotx = acx + atx;
                const atoty = acy + aty;
                
                if (phase !== 3) {
                    // --- PROYECCIONES GRISES ---
                    if (chk_ac.checked) {
                        drawProjection(px + acx, py + acy, -ur_x, -ur_y, "#aaa");
                    }
                    if (chk_at.checked && phase === 2) {
                        drawProjection(px + atx, py + aty, ut_x, ut_y, "#aaa");
                    }
                    if (chk_atot.checked && phase === 2) {
                        const normAtot = Math.sqrt(atotx*atotx + atoty*atoty);
                        if (normAtot > 0.1) {
                            drawProjection(px + atotx, py + atoty, atotx/normAtot, atoty/normAtot, "#aaa");
                        }
                    }

                    // --- VECTORES ---
                    if (chk_v.checked) drawArrow(ctx, px, py, px + vx, py + vy, "#20c997", 4);
                    if (chk_ac.checked && phase !== 3) drawArrow(ctx, px, py, px + acx, py + acy, "#dc3545", 4);
                    
                    if (chk_at.checked && phase === 2) {
                        drawArrow(ctx, px, py, px + atx, py + aty, "#fd7e14", 4);
                    }
                    
                    if (chk_atot.checked && phase === 2) {
                        drawArrow(ctx, px, py, px + atotx, py + atoty, "#6f42c1", 4);
                        
                        // Paralelogramo de suma
                        if (chk_ac.checked && chk_at.checked) {
                            drawArrow(ctx, px + acx, py + acy, px + atotx, py + atoty, "#ddd", 1, true);
                            drawArrow(ctx, px + atx, py + aty, px + atotx, py + atoty, "#ddd", 1, true);
                        }
                    }
                }

                // Dibujar partícula
                ctx.beginPath(); ctx.fillStyle = "#333";
                ctx.arc(px, py, 12, 0, Math.PI * 2); ctx.fill();

                requestAnimationFrame(animate);
            }
            
            btn_reset.addEventListener('click', () => {
                initAudio();
                isRunning = true;
                isPaused = false;
                phase = 1;
                theta = 0;
                omega = INITIAL_OMEGA;
                alpha = 0;
                
                btn_reset.innerText = "🔄 Reiniciar Animación";
                btn_pause.innerText = "⏸ Pausar";
                btn_pause.style.backgroundColor = "#ffc107";
                btn_pause.style.color = "black";
            });
            
            btn_pause.addEventListener('click', () => {
                if (!isRunning || phase === 3) return; // No pausar si no corre o ya terminó
                
                isPaused = !isPaused;
                if (isPaused) {
                    btn_pause.innerText = "▶ Continuar";
                    btn_pause.style.backgroundColor = "#0d6efd";
                    btn_pause.style.color = "white";
                    if (audioCtx && audioCtx.state === 'running') {
                        audioCtx.suspend();
                    }
                } else {
                    btn_pause.innerText = "⏸ Pausar";
                    btn_pause.style.backgroundColor = "#ffc107";
                    btn_pause.style.color = "black";
                    if (audioCtx && audioCtx.state === 'suspended' && !isMuted) {
                        audioCtx.resume();
                    }
                }
            });
            
            btn_mute.addEventListener('click', () => {
                isMuted = !isMuted;
                if (gainNode) {
                    // Si ya se detuvo (fase 3), mantenemos el mute en 0
                    gainNode.gain.value = (isMuted || phase === 3) ? 0 : 0.25;
                }
                btn_mute.innerText = isMuted ? "🔇 Activar Sonido" : "🔊 Silenciar";
                btn_mute.style.backgroundColor = isMuted ? "#dc3545" : "#6c757d";
            });
            
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.font = "18px sans-serif";
            ctx.fillStyle = "#666";
            ctx.textAlign = "center";
            ctx.fillText("Presiona Iniciar para comenzar", canvas.width/2, canvas.height/2);

            requestAnimationFrame(animate);
        }
        
        initSimulation();
    })();
</script>
"""

html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))

In [4]:
import random
from IPython.display import display, HTML

# Generamos un ID único para evitar conflictos si se ejecuta la celda varias veces
uid = str(random.randint(10000, 99999))

# --- INICIO DEL STRING HTML/JS ---
simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50; text-align: center;">Simulación: Patinador en Pista Semicircular</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px; text-align: center;">
        La simulación corre en <strong>cámara lenta</strong> para que observes los vectores aplicados en el centro de masa. Nota cómo en los extremos la <strong>aceleración tangencial</strong> es máxima, mientras que en el fondo la <strong>aceleración centrípeta</strong> es la predominante.
    </p>
    
    <div style="display: flex; justify-content: center; flex-wrap: wrap; gap: 15px; margin-bottom: 20px;">
        <button id="btn_reset_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; width: 180px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); font-size: 1.05em;">▶ Iniciar / Reiniciar</button>
        <button id="btn_pause_UID" style="background-color: #ffc107; color: black; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; width: 180px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); font-size: 1.05em;">⏸ Pausar</button>
        <button id="btn_mute_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; width: 150px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); font-size: 1.05em;">🔊 Silenciar</button>
    </div>
    
    <div style="display: flex; justify-content: center; margin-bottom: 15px; position: relative;">
        <canvas id="canvas_mcu_UID" width="600" height="400" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
    
    <div style="font-size: 0.95em; padding: 15px; background: #e2e3e5; border-radius: 5px; color: #383d41; display: flex; flex-wrap: wrap; gap: 15px; justify-content: center; user-select: none;">
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_v_UID" checked>
            <span><b style="color:#20c997;">▬ v</b> (Vel. Lineal)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_ac_UID">
            <span><b style="color:#dc3545;">▬ a_c</b> (Centrípeta)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_at_UID">
            <span><b style="color:#fd7e14;">▬ a_t</b> (Tangencial)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_atot_UID">
            <span><b style="color:#6f42c1;">▬ a_tot</b> (Total)</span>
        </label>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_mcu_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_reset = document.getElementById('btn_reset_UID');
            const btn_pause = document.getElementById('btn_pause_UID');
            const btn_mute = document.getElementById('btn_mute_UID');
            
            const chk_v = document.getElementById('chk_v_UID');
            const chk_ac = document.getElementById('chk_ac_UID');
            const chk_at = document.getElementById('chk_at_UID');
            const chk_atot = document.getElementById('chk_atot_UID');
            
            let lastTime = 0;
            let isRunning = false;
            let isPaused = false;
            let isMuted = false;
            
            const AudioContext = window.AudioContext || window.webkitAudioContext;
            let audioCtx = null;
            let oscillator = null;
            let filterNode = null;
            let gainNode = null;
            
            // --- CONSTANTES DE FÍSICA Y VISUALIZACIÓN ---
            const TIME_SCALE = 0.35; // Cámara lenta
            const R_visual = 160; 
            const R_fisico = 3.0; 
            const g_sim = 9.8; 
            
            let ang = Math.PI; // Inicia en la izquierda
            let omega = 0; 
            
            function initAudio() {
                if (!audioCtx) audioCtx = new AudioContext();
                if (audioCtx.state === 'suspended') audioCtx.resume();
                
                if (oscillator) {
                    oscillator.stop();
                    oscillator.disconnect();
                }
                
                oscillator = audioCtx.createOscillator();
                filterNode = audioCtx.createBiquadFilter();
                gainNode = audioCtx.createGain();
                
                oscillator.type = 'sawtooth'; 
                oscillator.frequency.value = 40; 
                filterNode.type = 'lowpass';
                filterNode.frequency.value = 100; 
                gainNode.gain.value = isMuted ? 0 : 0.25; 
                
                oscillator.connect(filterNode);
                filterNode.connect(gainNode);
                gainNode.connect(audioCtx.destination);
                oscillator.start();
            }

            function updateAudio(currentOmega) {
                if (oscillator && audioCtx && filterNode) {
                    const absOmega = Math.abs(currentOmega);
                    const targetFreq = 40 + (absOmega * 60);
                    const filterTarget = 100 + (absOmega * 80);
                    oscillator.frequency.setTargetAtTime(targetFreq, audioCtx.currentTime, 0.1);
                    filterNode.frequency.setTargetAtTime(filterTarget, audioCtx.currentTime, 0.1);
                }
            }

            function drawArrow(context, fromx, fromy, tox, toy, color, lineWidth=3, dashed=false) {
                const headlen = 10;
                const dx = tox - fromx;
                const dy = toy - fromy;
                const length = Math.sqrt(dx*dx + dy*dy);
                if(length < 0.1) return; 
                
                const angle = Math.atan2(dy, dx);
                context.beginPath();
                context.strokeStyle = color;
                context.lineWidth = lineWidth;
                if(dashed) context.setLineDash([5, 5]);
                else context.setLineDash([]);
                
                context.moveTo(fromx, fromy);
                context.lineTo(tox, toy);
                context.stroke();
                context.setLineDash([]); 
                
                context.beginPath();
                context.fillStyle = color;
                context.moveTo(tox, toy);
                context.lineTo(tox - headlen * Math.cos(angle - Math.PI / 6), toy - headlen * Math.sin(angle - Math.PI / 6));
                context.lineTo(tox - headlen * Math.cos(angle + Math.PI / 6), toy - headlen * Math.sin(angle + Math.PI / 6));
                context.fill();
            }

            function drawProjection(fromX, fromY, dirX, dirY, color) {
                const extLen = 300; 
                ctx.beginPath();
                ctx.strokeStyle = color;
                ctx.lineWidth = 1.5;
                ctx.setLineDash([4, 4]);
                ctx.moveTo(fromX - dirX * extLen, fromY - dirY * extLen);
                ctx.lineTo(fromX + dirX * extLen, fromY + dirY * extLen);
                ctx.stroke();
                ctx.setLineDash([]);
            }

            function animate(timestamp) {
                if (!isRunning || isPaused) {
                    lastTime = timestamp;
                    requestAnimationFrame(animate);
                    return;
                }
                
                if (!lastTime) lastTime = timestamp;
                let dt = ((timestamp - lastTime) / 1000) * TIME_SCALE;
                lastTime = timestamp;
                if(dt > 0.1) dt = 0.1; 
                
                const alpha = (g_sim / R_fisico) * Math.cos(ang);
                omega += alpha * dt;
                ang += omega * dt;

                if (ang > Math.PI) { ang = Math.PI; omega = -omega * 0.99; }
                if (ang < 0) { ang = 0; omega = -omega * 0.99; }

                ctx.clearRect(0, 0, canvas.width, canvas.height);
                const cx = canvas.width / 2;
                const cy = 80; 
                
                // 1. Tubería
                ctx.beginPath(); 
                ctx.strokeStyle = "#444"; 
                ctx.lineWidth = 5;
                ctx.arc(cx, cy, R_visual, 0, Math.PI, false); 
                ctx.stroke();

                // 2. Línea horizontal punteada
                ctx.beginPath();
                ctx.strokeStyle = "#999"; 
                ctx.lineWidth = 1.5;
                ctx.setLineDash([8, 8]);
                ctx.moveTo(cx - R_visual - 30, cy);
                ctx.lineTo(cx + R_visual + 30, cy);
                ctx.stroke();
                ctx.setLineDash([]);

                // Puntos de referencia
                function drawPoint(p_ang, label, align) {
                    const px = cx + R_visual * Math.cos(p_ang);
                    const py = cy + R_visual * Math.sin(p_ang);
                    ctx.beginPath(); ctx.fillStyle = "#333"; ctx.arc(px, py, 4, 0, Math.PI*2); ctx.fill();
                    ctx.font = "bold 14px sans-serif"; ctx.textAlign = align;
                    ctx.fillStyle = "#555"; 
                    ctx.fillText(label, px + (align === 'left' ? 10 : (align === 'right' ? -10 : 0)), py + 20);
                }
                drawPoint(Math.PI / 2, "Fondo", "center");
                drawPoint(37 * Math.PI / 180, "Punto P (37°)", "left"); 

                // Gravedad
                drawArrow(ctx, 40, 40, 40, 90, "#888", 2);
                ctx.font = "italic 18px serif"; ctx.textAlign = "left"; ctx.fillStyle = "#888"; 
                ctx.fillText("g", 50, 70);

                // --- VECTORES UNITARIOS Y POSICIÓN ---
                const ur_x = Math.cos(ang); 
                const ur_y = Math.sin(ang); 
                const ut_x = -Math.sin(ang); 
                const ut_y = Math.cos(ang); 

                // Punto exacto sobre el riel
                const px = cx + R_visual * ur_x; 
                const py = cy + R_visual * ur_y;

                // --- DIBUJAR PATINADOR ---
                // Ajustes para asentar ruedas y tabla encima de la pista
                const wheelOffset = 6.5; // Eleva el centro de la rueda respecto al riel (-ur)
                const boardOffset = 11.5; // Eleva la tabla
                
                const bx = px - boardOffset*ur_x;
                const by = py - boardOffset*ur_y;

                // Ruedas
                ctx.fillStyle = "#222";
                ctx.beginPath(); ctx.arc(px - 14*ut_x - wheelOffset*ur_x, py - 14*ut_y - wheelOffset*ur_y, 4, 0, Math.PI*2); ctx.fill();
                ctx.beginPath(); ctx.arc(px + 14*ut_x - wheelOffset*ur_x, py + 14*ut_y - wheelOffset*ur_y, 4, 0, Math.PI*2); ctx.fill();

                // Tabla
                ctx.beginPath(); ctx.strokeStyle = "#8B4513"; ctx.lineWidth = 4; ctx.lineCap = "round";
                ctx.moveTo(bx - 22*ut_x, by - 22*ut_y); ctx.lineTo(bx + 22*ut_x, by + 22*ut_y); ctx.stroke();

                // Esqueleto detallado
                ctx.strokeStyle = "#111"; ctx.lineWidth = 3; ctx.lineJoin = "round";
                const pelvisX = bx - 14*ur_x;
                const pelvisY = by - 14*ur_y;
                
                // Piernas flexionadas
                ctx.beginPath(); ctx.moveTo(bx - 12*ut_x, by); ctx.lineTo(bx - 16*ut_x - 7*ur_x, by - 7*ur_y); ctx.lineTo(pelvisX, pelvisY); ctx.stroke();
                ctx.beginPath(); ctx.moveTo(bx + 12*ut_x, by); ctx.lineTo(bx + 16*ut_x - 7*ur_x, by - 7*ur_y); ctx.lineTo(pelvisX, pelvisY); ctx.stroke();

                // Torso
                const shoulderX = pelvisX - 14*ur_x;
                const shoulderY = pelvisY - 14*ur_y;
                ctx.beginPath(); ctx.moveTo(pelvisX, pelvisY); ctx.lineTo(shoulderX, shoulderY); ctx.stroke();

                // Casco
                ctx.beginPath(); ctx.fillStyle = "#d35400"; // Naranja oscuro/rojo
                ctx.arc(shoulderX - 5*ur_x, shoulderY - 5*ur_y, 6.5, 0, Math.PI*2); ctx.fill(); ctx.stroke();

                // Brazos extendidos haciendo balance
                ctx.beginPath(); ctx.moveTo(shoulderX, shoulderY); ctx.lineTo(shoulderX - 14*ut_x + 4*ur_x, shoulderY - 14*ut_y + 4*ur_y); ctx.stroke();
                ctx.beginPath(); ctx.moveTo(shoulderX, shoulderY); ctx.lineTo(shoulderX + 14*ut_x + 4*ur_x, shoulderY + 14*ut_y + 4*ur_y); ctx.stroke();

                ctx.lineCap = "butt"; ctx.lineJoin = "miter"; // Restaurar estilos

                // --- CÁLCULO DE VECTORES ---
                const currentV = omega * R_fisico; 
                const ac = (currentV * currentV) / R_fisico; 
                const at_scalar = alpha * R_fisico; 

                // Escalas
                const scale_v = 12; 
                const scale_ac = 4.5; 
                const scale_at = 8; 

                // Origen de vectores en el Centro de Masa (Pelvis)
                const vox = pelvisX;
                const voy = pelvisY;

                const vx = ut_x * currentV * scale_v;
                const vy = ut_y * currentV * scale_v;
                
                const ac_arrow_x = -ur_x * ac * scale_ac; 
                const ac_arrow_y = -ur_y * ac * scale_ac;
                
                const at_arrow_x = ut_x * at_scalar * scale_at;
                const at_arrow_y = ut_y * at_scalar * scale_at;
                
                const atot_arrow_x = ac_arrow_x + at_arrow_x; 
                const atot_arrow_y = ac_arrow_y + at_arrow_y;
                
                // Dibujar Proyecciones Ejes
                if (chk_ac.checked) drawProjection(vox, voy, ur_x, ur_y, "#eee");
                if (chk_at.checked) drawProjection(vox, voy, ut_x, ut_y, "#eee");

                // Dibujar Vectores
                if (chk_v.checked) drawArrow(ctx, vox, voy, vox + vx, voy + vy, "#20c997", 4);
                if (chk_ac.checked) drawArrow(ctx, vox, voy, vox + ac_arrow_x, voy + ac_arrow_y, "#dc3545", 4);
                if (chk_at.checked) drawArrow(ctx, vox, voy, vox + at_arrow_x, voy + at_arrow_y, "#fd7e14", 4);
                if (chk_atot.checked) drawArrow(ctx, vox, voy, vox + atot_arrow_x, voy + atot_arrow_y, "#6f42c1", 4);
                
                // Paralelogramo
                if (chk_atot.checked && chk_ac.checked && chk_at.checked) {
                    drawArrow(ctx, vox + ac_arrow_x, voy + ac_arrow_y, vox + atot_arrow_x, voy + atot_arrow_y, "#ccc", 1, true);
                    drawArrow(ctx, vox + at_arrow_x, voy + at_arrow_y, vox + atot_arrow_x, voy + atot_arrow_y, "#ccc", 1, true);
                }

                if (isRunning && !isPaused) updateAudio(omega); 

                requestAnimationFrame(animate);
            }
            
            btn_reset.addEventListener('click', () => {
                initAudio();
                isRunning = true;
                isPaused = false;
                ang = Math.PI; 
                omega = 0;
                
                btn_reset.innerText = "🔄 Reiniciar Animación";
                btn_pause.innerText = "⏸ Pausar";
                btn_pause.style.backgroundColor = "#ffc107";
                btn_pause.style.color = "black";
            });
            
            btn_pause.addEventListener('click', () => {
                if (!isRunning) return; 
                
                isPaused = !isPaused;
                if (isPaused) {
                    btn_pause.innerText = "▶ Continuar";
                    btn_pause.style.backgroundColor = "#0d6efd";
                    btn_pause.style.color = "white";
                    if (audioCtx && audioCtx.state === 'running') audioCtx.suspend();
                } else {
                    btn_pause.innerText = "⏸ Pausar";
                    btn_pause.style.backgroundColor = "#ffc107";
                    btn_pause.style.color = "black";
                    if (audioCtx && audioCtx.state === 'suspended' && !isMuted) audioCtx.resume();
                }
            });
            
            btn_mute.addEventListener('click', () => {
                isMuted = !isMuted;
                if (gainNode) gainNode.gain.value = isMuted ? 0 : 0.25;
                btn_mute.innerText = isMuted ? "🔇 Activar Sonido" : "🔊 Silenciar";
                btn_mute.style.backgroundColor = isMuted ? "#dc3545" : "#6c757d";
            });
            
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.font = "18px sans-serif";
            ctx.fillStyle = "#666";
            ctx.textAlign = "center";
            ctx.fillText("Presiona Iniciar para ver al patinador en acción", canvas.width/2, canvas.height/2);

            requestAnimationFrame(animate);
        }
        
        initSimulation();
    })();
</script>
"""
# --- FIN DEL STRING HTML/JS ---

# --- CÓDIGO PYTHON PARA EJECUTAR LA SIMULACIÓN ---
html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))